In [ ]:
# 사전 준비
'''
../assignment1/output/curated/alert_lifetime_summary.csv 파일을
현 디렉토리의 data 폴더 안에 같은 형태로 넣어주세요.
이전에 활용한 CustomerLoyaltyProgram.csv 파일도 넣어주세요.
'''

In [ ]:
# 시나리오 설명
'''
Assignment 1에서 생성된 고객 알림 데이터에는 고객 ID와 alert 여부 정보만 포함되어 있습니다.
김싸피가 속한 마케팅 팀은 각 국가별로 알림 대상 고객을 별도 파일로 분리하여 마케팅 담당자에게 전달하고자 합니다.
이를 위해 고객 원본 데이터에서 지역 정보를 병합하고, alert == "yes"인 고객을 국가 단위로 분할하여 저장하세요.
'''

In [ ]:
# 요구 사항
'''
assignment1의 ../output/curated/alert_lifetime_summary.csv 파일을 불러옵니다.
CustomerLoyaltyProgram.csv 파일도 data 폴더에 위치합니다.
Loyalty# 기준으로 지역 정보를 병합합니다,
alert == "yes" 조건으로 필터링합니다.
국가(Country) 기준으로 그룹핑합니다.
../output/serving/reminder_by_region/ 폴더에 국가별 CSV 저장합니다.
'''

In [6]:
import os
import pandas as pd

SUMMARY_PATH = "../output/curated/alert_lifetime_summary.csv"
REGION_PATH = "../data/CustomerLoyaltyProgram.csv"
OUTPUT_DIR = "../output/serving/reminder_by_region"


In [7]:
def extract_summary(path: str) -> pd.DataFrame:
    """
    Extract 단계: Job1 결과 로드
    """
    df = pd.read_csv(path)
    return df

In [8]:
def extract_region(path: str) -> pd.DataFrame:
    """
    Extract 단계: 원천 데이터에서 지역 정보만 추출
    - 필요한 컬럼: Loyalty#, Country, Province or State, City
    - 중복 제거 필요
    """
    df = pd.read_csv(path)

    region_info = (
        df[["Loyalty#", "Country", "Province or State", "City"]]
        .drop_duplicates()
    )
    return region_info

In [9]:
def transform_merge_and_filter(summary_df: pd.DataFrame, region_df: pd.DataFrame) -> pd.DataFrame:
    """
    Transform 단계:
    - Loyalty# 기준 merge
    - alert == "yes" 인 데이터만 필터링
    """
    merged = summary_df.merge(region_df, on="Loyalty#", how="inner")

    alert_df = merged[merged["alert"] == "yes"].reset_index(drop=True)
    return alert_df

In [12]:
def load_by_country(df: pd.DataFrame, out_dir: str):
    """
    Load 단계:
    - Country 기준으로 그룹핑
    - 국가별 CSV 파일 저장
    """
    os.makedirs(out_dir, exist_ok=True)

    # Country 결측치 처리
    df["Country"] = df["Country"].fillna("Unknown")

    for country, group in df.groupby("Country"):
        safe_country = str(country).strip().replace(" ", "_")
        filename = f"reminder_{safe_country}.csv"
        save_path = os.path.join(out_dir, filename)

        group.to_csv(save_path, index=False)
        print(f"[{country}] 저장 완료 - {len(group)}명 -> {save_path}")

In [13]:
summary_df = extract_summary(SUMMARY_PATH)
region_df = extract_region(REGION_PATH)
alert_df = transform_merge_and_filter(summary_df, region_df)
load_by_country(alert_df, OUTPUT_DIR)

print("Job2 완료: 국가별 reminder 파일 생성")
print(alert_df.head())


[Canada] 저장 완료 - 251명 -> ../output/serving/reminder_by_region\reminder_Canada.csv
[Germany] 저장 완료 - 241명 -> ../output/serving/reminder_by_region\reminder_Germany.csv
[United Kingdom] 저장 완료 - 229명 -> ../output/serving/reminder_by_region\reminder_United_Kingdom.csv
[United States] 저장 완료 - 291명 -> ../output/serving/reminder_by_region\reminder_United_States.csv
Job2 완료: 국가별 reminder 파일 생성
   Loyalty#      Customer Name  TotalLifetimeValue alert         Country  \
0  100616.0      Melodi Choate            61850.19   yes         Germany   
1  100823.0       Keturah Haas           122269.36   yes  United Kingdom   
2  101544.0    Travis Hutching            64839.92   yes   United States   
3  101544.0    Travis Hutching            64839.92   yes          Canada   
4  105490.0  Donetta Vanmarter            51016.07   yes         Germany   

     Province or State           City  
0               Bayern       Nurnberg  
1       Greater London         London  
2           California  San Francis